In [ ]:
import optuna
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from data_processing import create_descriptors

### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

# Define target here
target = "Density"
# Define path to csv

def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            'MolWt': Descriptors.MolWt(mol),
            'NumAtoms': mol.GetNumAtoms(),
            'LogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'RingCount': Descriptors.RingCount(mol),
            'Flexibility': Descriptors.NumRotatableBonds(mol) / max(1, mol.GetNumAtoms()),
            'HeavyAtoms': Descriptors.HeavyAtomCount(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ["Density"] # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns]
    else:
        features = data
        target = None

    return features, target

train_path = "/home/stas/Documents/GitHub/FEDOT.LLM/examples/polymers_predict/polymer_Density/competition/train.csv"
test_path = "/home/stas/Documents/GitHub/FEDOT.LLM/examples/polymers_predict/polymer_Density/competition/test.csv"
train = pd.read_csv(train_path)
X_test = pd.read_csv(test_path)


/home/stas/Documents/GitHub/FEDOT.LLM/.venv/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [125]:
EVAL_SET_SIZE = 0.2
train_data, eval_test_data = train_test_split(
    train, test_size=EVAL_SET_SIZE, random_state=42
)  # corresponding to 80%, 20% of ‘dataset‘

train_features, train_target = transform_data(train_data)
eval_test_features, eval_test_target = transform_data(eval_test_data)
test_features, _ = transform_data(X_test)

In [ ]:
def optimize_booster(X_train, y_train, X_val, y_val, target_name):
    def objective(trial):
        model_type = trial.suggest_categorical("model", ["catboost", "lgbm", "xgb"])

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
            "learning_rate": trial.suggest_float("lr", 0.005, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 16),
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        }

        if model_type == "catboost":
            model = CatBoostRegressor(**params, silent=True)
        elif model_type == "lgbm":
            model = LGBMRegressor(**params)
        else:
            model = XGBRegressor(**params)

        model.fit(X_train, y_train[target_name])
        return mean_absolute_error(y_val[target_name], model.predict(X_val))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=2)
    return study.best_params


print(f"\n=== Training {target} model ===")
best_params = optimize_booster(X_train, y_train, X_val, y_val, target)

if best_params["model"] == "catboost":
    model = CatBoostRegressor(
        iterations=best_params["n_estimators"],
        learning_rate=best_params["lr"],
        depth=best_params["max_depth"],
        subsample=best_params["subsample"],
        # colsample_bylevel=best_params['colsample'],
        loss_function="MAE",
        verbose=False,
    )
elif best_params["model"] == "lgbm":
    model = LGBMRegressor(
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["lr"],
        max_depth=best_params["max_depth"],
        subsample=best_params["subsample"],
        # colsample_bytree=best_params['colsample'],
        objective="mae",
        random_state=42,
    )
else:
    model = XGBRegressor(
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["lr"],
        max_depth=best_params["max_depth"],
        subsample=best_params["subsample"],
        # colsample_bytree=best_params['colsample'],
        eval_metric="mae",
        random_state=42,
    )

model.fit(
    X_train,
    y_train[target],
)

preds = model.predict(X_test)
print("MAE", mean_absolute_error(y_test[target], preds))
print("R2", model.score(X_test, y_test[target]))

<class 'pandas.core.frame.DataFrame'>
2025-07-23 20:26:20,743 - ApiComposer - Initial pipeline was fitted in 3.8 sec.
2025-07-23 20:26:20,755 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 19.0 sec.
2025-07-23 20:26:20,757 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-23 20:26:20,767 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1.0 min. Set of candidate models: ['adareg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ransac_lin_reg', 'rfr', 'ridge', 'scaling'].
2025-07-23 20:26:20,806 - ApiComposer - Timeout is too small for composing and is skipped because fit_time is 18.993945 sec.
2025-07-23 20:26:20,814 - ApiComposer - Hyperparameters tuning started with 1 min. timeout
2025-07-23 20:26:39,927 - SimultaneousTuner - Initial graph: {'depth': 2, 'length': 2, 'nodes': [rfr, scaling]}
rfr - {'n_jobs': 1}
scaling - {} 
Initial metric: [0.623]
  0%

In [138]:
def evaluate_model(model, test_features: pd.DataFrame | pd.Series, test_target: pd.DataFrame | pd.Series):
    input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()

def automl_predict(model, features: pd.DataFrame | pd.Series) -> pd.Series:
    input_data = InputData.from_dataframe(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions


# --- Step 4: Evaluate the trained model ---
evaluate_model(model, eval_test_features, eval_test_target)

Model metrics:  {'r2': 0.43, 'mae': 0.063}


{'r2': 0.43, 'mae': 0.063}

In [139]:
res = automl_predict(model, test_features)
print(res)

Predictions shape: (3,)
[1.07041955 1.29533629 1.13359502]


In [2]:
import optuna
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from data_processing import create_descriptors

from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import Task
from fedot.core.repository.tasks import TaskTypesEnum # classification, regression, ts_forecasting.

# Define target here
target = "Density"
# Define path to csv
path = "competition/train.csv"

data = pd.read_csv(path)
X, y = create_descriptors(data)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

def train_model(train_features: pd.DataFrame | pd.Series, train_target: pd.DataFrame | pd.Series):
    print(type(train_features))
    input_data = InputData.from_dataframe(train_features, train_target, task='classification')
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1.0,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric=['r2', 'mae'],
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    model.fit(features=input_data) # this is the training step, after this step variable ‘model‘ will be a trained model

    # Save the pipeline
    pipeline = model.current_pipeline
    #pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
  
def evaluate_model(model, test_features: pd.DataFrame | pd.Series, test_target: pd.DataFrame | pd.Series):
    input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()

def automl_predict(model, features: pd.DataFrame | pd.Series) -> pd.Series:
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")


ValueError: No valid samples left after filtering NaN targets

In [2]:
X_train

,MolWt,NumAtoms,LogP,TPSA,RingCount,Flexibility,HeavyAtoms,FP_0,FP_1,FP_2,...,FP_1270,FP_1271,FP_1272,FP_1273,FP_1274,FP_1275,FP_1276,FP_1277,FP_1278,FP_1279
4756,452.720,34,8.86710,52.60,0,0.794118,32,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4470,98.189,9,2.72800,0.00,0,0.222222,7,0,1,0,...,0,1,0,0,0,0,0,0,0,0
1362,221.212,18,2.33350,69.44,1,0.277778,16,0,0,0,...,0,1,0,0,0,0,0,0,0,0
936,338.576,26,7.48660,26.30,0,0.730769,24,0,0,0,...,0,1,0,0,0,0,0,0,1,0
293,466.747,35,9.25720,52.60,0,0.800000,33,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,70.091,7,0.81010,17.07,0,0.142857,5,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1164,96.173,9,2.57740,0.00,0,0.555556,7,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3587,393.439,31,4.56528,85.62,2,0.387097,29,0,1,0,...,0,1,0,0,0,0,0,0,0,0
5594,141.170,12,0.15060,29.54,1,0.166667,10,0,1,0,...,0,1,0,0,0,0,0,0,0,0


In [6]:
# --- Step 3: Train AutoML model ---
model = train_model(X_train, y_train)

# --- Step 4: Evaluate the trained model ---
evaluate_model(model, X_val, y_val)




<class 'pandas.core.frame.DataFrame'>
2025-07-23 00:48:53,204 - ApiComposer - Initial pipeline was fitted in 4.5 sec.
2025-07-23 00:48:53,214 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 22.5 sec.
2025-07-23 00:48:53,215 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-23 00:48:53,224 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1.0 min. Set of candidate models: ['adareg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ransac_lin_reg', 'rfr', 'ridge', 'scaling'].
2025-07-23 00:48:53,257 - ApiComposer - Timeout is too small for composing and is skipped because fit_time is 22.5468 sec.
2025-07-23 00:48:53,265 - ApiComposer - Hyperparameters tuning started with 1 min. timeout
2025-07-23 00:49:15,164 - SimultaneousTuner - Initial graph: {'depth': 2, 'length': 2, 'nodes': [rfr, scaling]}
rfr - {'n_jobs': 1}
scaling - {} 
Initial metric: [0.568]
  0%| 

{'r2': 0.655, 'mae': 0.045}